# ASR Error Correction & Translation Pipeline

This notebook fine-tunes a T5 model to correct English ASR (speech-to-text) errors, and
evaluates the effect of that correction on downstream English→Chinese translation quality.

**Pipeline overview**
1. Load audio + reference transcripts + reference translations (CoVoST-style dataset)
2. Run `wav2vec2` ASR on the audio to get raw (noisy) transcripts
3. Fine-tune a `T5` model to correct ASR errors (`train` split)
4. Translate the corrected / original text with a MarianMT model
5. Score everything with WER, CER, and METEOR, comparing original vs. corrected

> Originally a local research script — restructured here into notebook cells so each stage
> can be run and inspected independently. Paths are configurable in the **Config** cell below
> instead of being hardcoded, so this runs on any machine / Colab.


## 0. Setup

Install dependencies (skip if already installed / running locally with `requirements.txt`).

In [ ]:
# Uncomment to install dependencies (e.g. on Google Colab)
# !pip install -q torch torchaudio transformers python-Levenshtein nltk jieba sacrebleu


In [ ]:
import os
import re
import string
from collections import Counter, defaultdict

import Levenshtein as lev
import torch
import torchaudio
import jieba
import nltk
from torch.utils.data import DataLoader, TensorDataset
from transformers import (
    Wav2Vec2Processor, Wav2Vec2ForCTC,
    T5Tokenizer, T5ForConditionalGeneration,
    MarianTokenizer, MarianMTModel,
)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score


In [ ]:
# Initialize NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)


## 1. Config

Set your local paths here. The original script hardcoded a personal `/Users/...` path —
this version reads from a `DATA_ROOT` variable so it works on any machine.
Expected folder layout:

```
DATA_ROOT/
  train/audio/*.mp3        train/transcripts/*.txt        train/translations/*.txt
  test/audio/*.mp3         test/transcripts/*.txt         test/translations/*.txt
```


In [ ]:
# ======================
# Config — edit this for your environment
# ======================
DATA_ROOT = os.environ.get("DATA_ROOT", "./data/CoVoST/small")

TRAIN_AUDIO_DIR = os.path.join(DATA_ROOT, "train", "audio")
TRAIN_TRANSCRIPTS_DIR = os.path.join(DATA_ROOT, "train", "transcripts")
TRAIN_TRANSLATIONS_DIR = os.path.join(DATA_ROOT, "train", "translations")

TEST_AUDIO_DIR = os.path.join(DATA_ROOT, "test", "audio")
TEST_TRANSCRIPTS_DIR = os.path.join(DATA_ROOT, "test", "transcripts")
TEST_TRANSLATIONS_DIR = os.path.join(DATA_ROOT, "test", "translations")

NUM_EPOCHS = 10
BATCH_SIZE = 8


## 2. Load models

ASR model (`wav2vec2`) and English→Chinese translation model (MarianMT).

In [ ]:
# ASR model
asr_processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-large-960h")
asr_model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-large-960h")
asr_model.eval()


In [ ]:
# Translation model (English -> Chinese)
translation_tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-zh")
translation_model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-zh")


def translate_text_with_punctuation(input_text):
    """Translate English text to Chinese and normalize punctuation."""
    inputs = translation_tokenizer(input_text, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        translated_ids = translation_model.generate(**inputs)
    raw_translation = translation_tokenizer.decode(translated_ids[0], skip_special_tokens=True)

    punctuation_map = {
        '.': '。', ',': '，', '?': '？', '!': '！', ':': '：', ';': '；'
    }

    end_punct = ''
    if len(input_text) > 0:
        last_char = input_text[-1]
        end_punct = punctuation_map.get(last_char, '')

    processed_translation = raw_translation
    for eng_punct, chi_punct in punctuation_map.items():
        processed_translation = processed_translation.replace(eng_punct, chi_punct)

    processed_translation = re.sub(r'\s+', '，', processed_translation)
    processed_translation = re.sub(r'，+', '，', processed_translation)

    if processed_translation and processed_translation[-1] in punctuation_map.values():
        processed_translation = processed_translation[:-1]
    processed_translation += end_punct

    processed_translation = processed_translation.replace('，。', '。')
    processed_translation = re.sub(r'([。！？])，', r'\1', processed_translation)

    return processed_translation


## 3. Data loading

In [ ]:
def load_aligned_data(audio_folder, transcripts_folder, translations_folder):
    """Load audio, transcripts, and translations, keeping them strictly aligned by filename."""
    audio_files = sorted([f for f in os.listdir(audio_folder) if f.endswith('.mp3')])
    audio_paths = []
    transcripts = []
    translations = []

    for audio_file in audio_files:
        base_name = os.path.splitext(audio_file)[0]
        audio_path = os.path.join(audio_folder, audio_file)
        audio_paths.append(audio_path)

        transcript_path = os.path.join(transcripts_folder, f"{base_name}.txt")
        if os.path.exists(transcript_path):
            with open(transcript_path, 'r', encoding='utf-8') as f:
                transcript = f.read().strip()
                if transcript and transcript[-1] not in {'.', '?', '!', ',', ';', ':', '"'}:
                    transcript += '.'
                transcripts.append(transcript)
        else:
            transcripts.append("")

        translation_path = os.path.join(translations_folder, f"{base_name}.txt")
        if os.path.exists(translation_path):
            with open(translation_path, 'r', encoding='utf-8') as f:
                translation = f.read().strip()
                translations.append(translation)
        else:
            translations.append("")
    return audio_paths, transcripts, translations


## 4. ASR generation

In [ ]:
def get_similarity(word1, word2):
    """Word similarity based on Levenshtein distance."""
    lev_distance = lev.distance(word1, word2)
    max_len = max(len(word1), len(word2))
    similarity = 1 - lev_distance / max_len
    return similarity


def clean_word(word):
    """Strip punctuation and a trailing 's'/'d' from a word."""
    word = re.sub(f'[{string.punctuation}]', '', word).lower()
    if word.endswith('s'):
        word = word[:-1]
    elif word.endswith('d'):
        word = word[:-1]
    return word


def is_suffix_variant(word1, word2):
    """Check whether two words differ only by a suffix (ing, er, ed, s)."""
    suffixes = ["ing", "er", "ed", "s"]
    for suffix in suffixes:
        if word1 == word2 + suffix:
            return True
        if word2 == word1 + suffix:
            return True
    return False


In [ ]:
def generate_asr_results(audio_paths, target_transcripts):
    asr_texts = []
    cleaned_transcripts = []
    mismatches = []
    valid_audio_paths = []
    valid_indices = []
    error_pairs = []
    global_error_pairs = defaultdict(int)

    for idx, (audio_path, transcript) in enumerate(zip(audio_paths, target_transcripts)):
        try:
            waveform, sample_rate = torchaudio.load(audio_path)
            if sample_rate != 16000:
                waveform = torchaudio.transforms.Resample(sample_rate, 16000)(waveform)
            if waveform.size(0) > 1:
                waveform = waveform.mean(dim=0, keepdim=True)
            if waveform.size(1) == 0:
                print(f"Warning: {audio_path} is empty, skipping")
                continue

            inputs = asr_processor(waveform.squeeze(), sampling_rate=16000, return_tensors="pt", padding=True)
            with torch.no_grad():
                logits = asr_model(**inputs).logits
            predicted_ids = torch.argmax(logits, dim=-1)
            asr_result = asr_processor.decode(predicted_ids[0]).lower().strip()

            asr_texts.append(asr_result)
            cleaned_transcripts.append(transcript.lower().strip())
            valid_audio_paths.append(audio_path)
            valid_indices.append(idx)

            if len(asr_result.split()) == len(transcript.split()):
                mismatch_count = sum(
                    lev.distance(a, t)
                    for a, t in zip(asr_result.split(), transcript.lower().split())
                )
                mismatches.append((asr_result, transcript, mismatch_count))

                for asr_w, trans_w in zip(asr_result.split(), transcript.lower().split()):
                    asr_clean = clean_word(asr_w)
                    trans_clean = clean_word(trans_w)
                    similarity = get_similarity(asr_clean, trans_clean)

                    if is_suffix_variant(asr_clean, trans_clean):
                        continue

                    if 0.75 < similarity < 1:
                        pair = f"{asr_w}--->{trans_w}"
                        error_pairs.append(pair)
                        global_error_pairs[pair] += 1
            else:
                print(f"Warning: {audio_path} ASR word count differs from reference, skipping similarity calc")

        except Exception as e:
            print(f"Error processing {audio_path}: {str(e)}")
            continue

    if error_pairs:
        print("Detected error pairs:")
        for pair, count in global_error_pairs.items():
            print(f"{pair} (count: {count})")

    return asr_texts, cleaned_transcripts, mismatches, valid_audio_paths, valid_indices, global_error_pairs


## 5. Text processing helpers

In [ ]:
def clean_text(text):
    """Clean ASR text."""
    return re.sub(r'\s+', ' ', text).replace(":", "").strip().lower()


def clean_translation(text):
    """Keep only Chinese characters in translation text."""
    return re.sub(r'[^\u4e00-\u9fa5]', '', text.strip())


def chinese_tokenize(text):
    """Chinese word segmentation."""
    return list(jieba.cut(text))


## 6. Evaluation metrics

In [ ]:
def calculate_wer(reference, hypothesis):
    """Word Error Rate."""
    ref_words = reference.split()
    hyp_words = hypothesis.split()
    if len(ref_words) == 0:
        return 0.0 if len(hyp_words) == 0 else 1.0
    distance = lev.distance(ref_words, hyp_words)
    return distance / len(ref_words)


def calculate_cer(reference, hypothesis):
    """Character Error Rate."""
    ref_chars = list(reference)
    hyp_chars = list(hypothesis)
    if len(ref_chars) == 0:
        return 0.0 if len(hyp_chars) == 0 else 1.0
    distance = lev.distance(ref_chars, hyp_chars)
    return distance / len(ref_chars)


## 7. Load train & test data, run baseline ASR

In [ ]:
train_audio, train_trans, train_translations = load_aligned_data(
    TRAIN_AUDIO_DIR, TRAIN_TRANSCRIPTS_DIR, TRAIN_TRANSLATIONS_DIR
)

train_asr, train_clean, train_mismatches, valid_train_audio, train_indices, train_error_pairs = \
    generate_asr_results(train_audio, train_trans)


In [ ]:
test_audio, test_trans, test_translations = load_aligned_data(
    TEST_AUDIO_DIR, TEST_TRANSCRIPTS_DIR, TEST_TRANSLATIONS_DIR
)

test_asr, test_clean, test_mismatches, valid_test_audio, test_indices, test_error_pairs = \
    generate_asr_results(test_audio, test_trans)

test_translations_valid = [test_translations[i] for i in test_indices]


## 8. Fine-tune the ASR error-correction model (T5)

In [ ]:
task_prefix = "fix ASR errors: "
tokenizer = T5Tokenizer.from_pretrained("t5-small")
model = T5ForConditionalGeneration.from_pretrained("t5-base")

train_inputs = [task_prefix + clean_text(t) for t in train_asr]
train_labels = [clean_text(t) for t in train_clean]

train_encodings = tokenizer(
    train_inputs, max_length=128, padding='max_length', truncation=True, return_tensors="pt"
)
train_label_encodings = tokenizer(
    train_labels, max_length=128, padding='max_length', truncation=True, return_tensors="pt"
)

train_dataset = TensorDataset(
    train_encodings.input_ids,
    train_encodings.attention_mask,
    train_label_encodings.input_ids,
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)
model.train()
for epoch in range(NUM_EPOCHS):
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids, attention_mask, labels = batch

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch + 1} - avg loss: {total_loss / len(train_loader):.4f}")


## 9. Evaluate: original ASR vs. corrected ASR (WER / CER / METEOR)

In [ ]:
smoothing = SmoothingFunction().method1

test_inputs = [task_prefix + clean_text(t) for t in test_asr]
test_encodings = tokenizer(
    test_inputs, max_length=128, padding='max_length', truncation=True, return_tensors="pt"
)

bad_words = [":", "english", "french", "german"]
bad_words_ids = tokenizer(bad_words, add_special_tokens=False).input_ids
bad_words_ids = [item for sublist in bad_words_ids for item in sublist]


In [ ]:
total_samples = 0
total_cer_original = 0.0
total_cer_corrected = 0.0
total_wer_original = 0.0
total_wer_corrected = 0.0
total_meteor_original = 0.0
total_meteor_corrected = 0.0

model.eval()
with torch.no_grad():
    for batch_idx in range(0, len(test_encodings.input_ids), BATCH_SIZE):
        batch_input_ids = test_encodings.input_ids[batch_idx:batch_idx + BATCH_SIZE]
        batch_attention_mask = test_encodings.attention_mask[batch_idx:batch_idx + BATCH_SIZE]

        generated_ids = model.generate(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_mask,
            max_length=128,
            num_beams=4,
            no_repeat_ngram_size=2,
            bad_words_ids=[bad_words_ids],
            early_stopping=True,
        )

        predictions = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

        for i, pred in enumerate(predictions):
            actual_idx = batch_idx + i
            if actual_idx >= len(test_clean):
                break

            original_asr = test_asr[actual_idx]
            ground_truth = test_clean[actual_idx]
            reference_trans = test_translations_valid[actual_idx]
            cleaned_pred = pred.replace("fix ASR errors:", "").strip()

            wer_original = calculate_wer(ground_truth, original_asr)
            total_wer_original += wer_original

            wer_corrected = calculate_wer(ground_truth, cleaned_pred)
            total_wer_corrected += wer_corrected

            cer_original = calculate_cer(ground_truth, original_asr)
            total_cer_original += cer_original

            cer_corrected = calculate_cer(ground_truth, cleaned_pred)
            total_cer_corrected += cer_corrected

            try:
                translated_pred = translate_text_with_punctuation(cleaned_pred)
            except Exception as e:
                print(f"Translation failed: {str(e)}")
                translated_pred = ""

            try:
                translated_original = translate_text_with_punctuation(original_asr)
            except Exception as e:
                print(f"Original ASR translation failed: {str(e)}")
                translated_original = ""

            clean_ref = clean_translation(reference_trans)
            clean_pred = clean_translation(translated_pred)
            clean_original_trans = clean_translation(translated_original)

            if clean_ref and clean_pred:
                ref_tokens = chinese_tokenize(clean_ref)
                pred_tokens = chinese_tokenize(clean_pred)
                meteor_val_corrected = meteor_score([ref_tokens], pred_tokens)
                total_meteor_corrected += meteor_val_corrected
            else:
                meteor_val_corrected = 0.0

            if clean_ref and clean_original_trans:
                ref_tokens = chinese_tokenize(clean_ref)
                original_tokens = chinese_tokenize(clean_original_trans)
                meteor_val_original = meteor_score([ref_tokens], original_tokens)
                total_meteor_original += meteor_val_original
            else:
                meteor_val_original = 0.0

            print(f"Sample {actual_idx + 1}/{len(test_clean)}")
            print(f"Original ASR: {original_asr}")
            print(f"Corrected: {cleaned_pred}")
            print(f"Reference: {ground_truth}")
            print(f"Original translation: {translated_original}")
            print(f"Corrected translation: {translated_pred}")
            print(f"Reference translation: {reference_trans}")
            print(f"Original ASR CER: {cer_original:.4f}")
            print(f"Corrected CER: {cer_corrected:.4f}")
            print(f"Original ASR WER: {wer_original:.4f}")
            print(f"Corrected WER: {wer_corrected:.4f}")
            print(f"Original ASR METEOR: {meteor_val_original:.4f}")
            print(f"Corrected METEOR: {meteor_val_corrected:.4f}")
            print("=" * 80)

            total_samples += 1


In [ ]:
if total_samples > 0:
    print("\nFinal evaluation results:")
    print(f"Test samples: {total_samples}")
    print(f"Avg original ASR CER: {total_cer_original / total_samples:.4f}")
    print(f"Avg corrected CER: {total_cer_corrected / total_samples:.4f}")
    print(f"Avg original ASR WER: {total_wer_original / total_samples:.4f}")
    print(f"Avg corrected WER: {total_wer_corrected / total_samples:.4f}")
    print(f"Avg original ASR METEOR: {total_meteor_original / total_samples:.4f}")
    print(f"Avg corrected METEOR: {total_meteor_corrected / total_samples:.4f}")
else:
    print("Error: no valid test samples")
